In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
import sys
import pandas as pd
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.query import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *


EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully


In [3]:
output_dict={
    'ReportAssetLengthKm': ['Km', 'Total length of all the assets in the specified period', ''],
    'AssetCoveredLengthKm': ['Km', 'Total length of all the assets covered in the report', ''],
    'DistributionPipeKm': ['Km', 'Total length of all the mains in the report', ''],
    'DistributionPipeCoveredKm': ['Km', 'Total length of all the mains covered in the report', ''],
    'ServicePipeKm': ['Km', 'Total length of all the services in the report', ''],
    'ServicePipeCoveredKm': ['Km', 'Total length of all the services covered in the report', ''],
    'ReportCount': ['Report', 'Total number of reports in the specified week', ''],
    'DaysCount': ['Day', 'Total number of days worked in the specified week', ''],
    'FOVMain': ['Percent', 'Km of mains covered over the total Km of mains', 'DistributionPipeCoveredKm / DistributionPipeKm'],
    'SurveyDurationHours': ['Hours', 'Total hours driven by all cars in the week', ''],
    'TargetDurationHours': ['Hours', 'Exepected total number hours driven by all cars available. i.e 7 days * 7 hours * 25 cars', ''],
    'CustomerUtilization': ['Percent', 'Total hours driven by all cars in the week over the expected total number hours driven by all cars available. i.e 7 days * 7 hours * 25 cars', 'SurveyDurationHours / TargetDurationHours'],
    'StarndardUtilization': ['Percent', 'Total hours driven by all cars in the week over the expected total number hours driven by all cars available. i.e 6 days * 5 hours * 25 cars', 'SurveyDurationHours / TargetDurationHours'],
    'TotalSurveyors': ['Surveyor', 'Total number of surveyors in the specified week', ''],
    'ProductivityPerSurveyor': ['Km / Surveyor', 'Total length of all the assets covered in the report over the total number of surveyors in the specified week', 'DistributionPipeCoveredKm / TotalSurveyors'],
    'SurveyCount': ['Survey', 'Total number of surveys in the  specified week', ''],
    'AvgSpeedKm': ['Km/h', 'Average speed counting all the cars in the specified week', ''],
    'SurveysCarDay': ['Survey / Car / Day', 'Total number of surveys per car per day in the specified week', 'SurveyCount / TotalSurveyors / DaysCount'],
    'IdleTime': ['Percent', 'Ratio of idle time over the total survey duration', 'IdleTimeMinutes / SurveyDurationMinutes'],
    'TotalDrivenLengthKm': ['Km', 'Total length driven in the specified week', ''],
    'DrivingRatio': ['Ratio', 'Total length driven in the specified week over Total length of all the assets covered in the report', 'TotalKilometers / DistributionPipeCoveredKm'],
    'NightDrivenLength': ['Km', 'Total length covered during the night in the specified week', ''],
    'DayDrivenLength': ['Km', 'Total length covered during the day in the specified week', ''],
    'NightRatio': ['Ratio', 'NightKm / TotalKilometers', ''],
    'DayRatio': ['Ratio', 'DayKm / TotalKilometers', ''],
    'LisaCount': ['Lisa', 'Total number of Lisa in the specified week, (No Disposition 2)', ''],
    'EmissionRate': ['SCFH', 'Total emission rate in the specified week, (No Disposition 2)', ''],
    'B0Count': ['Lisa', 'Total number of B0 in the specified week, (No Disposition 2)', ''],
    'B1Count': ['Lisa', 'Total number of B1 in the specified week, (No Disposition 2)', ''],
    'Bm1Count': ['Lisa', 'Total number of B-1 in the specified week, (No Disposition 2)', ''],
    'Bm2Count': ['Lisa', 'Total number of B-2 in the specified week, (No Disposition 2)', ''],
    'NGCount': ['Lisa', 'Total number of NG in the specified week, (With Disposition 2)', ''],
    'PGCount': ['Lisa', 'Total number of PG in the specified week, (With Disposition 2)', ''],
    'Not_NGCount': ['Lisa', 'Total number of Not NG in the specified week, (With Disposition 2)', ''],
    'LisaDensity': ['Lisa / Km', 'Number of Lisa per Km of Asset Covered', 'LisaCount / DistributionPipeCoveredKm'],
    'InstatanoeusEmission': ['SCFH / Km', 'Emission rate per Km of Asset Covered', 'EmissionRate / DistributionPipeCoveredKm'],
    'B0Density': ['Lisa / Km', 'Number of B0 per Km of Asset Covered', 'B0Count / DistributionPipeCoveredKm'],
    'B1Density': ['Lisa / Km', 'Number of B1 per Km of Asset Covered', 'B1Count / DistributionPipeCoveredKm'],
    'Bm1Density': ['Lisa / Km', 'Number of B-1 per Km of Asset Covered', 'Bm1Count / DistributionPipeCoveredKm'],
    'Bm2Density': ['Lisa / Km', 'Number of B-2 per Km of Asset Covered', 'Bm2Count / DistributionPipeCoveredKm'],
    'B0Share': ['Percent', 'Number of B0 per Lisa', 'B0Count / LisaCount'],
    'B1Share': ['Percent', 'Number of B1 per Lisa', 'B1Count / LisaCount'],
    'Bm1Share': ['Percent', 'Number of B-1 per Lisa', 'Bm1Count / LisaCount'],
    'Bm2Share': ['Percent', 'Number of B-2 per Lisa', 'Bm2Count / LisaCount'],
    'NGShare': ['Percent', 'Number of NG per Lisa', 'NGCount / (NGCount + Not_NGCount + PGCount)'],
    'PGShare': ['Percent', 'Number of PG per Lisa', 'PGCount / (NGCount + Not_NGCount + PGCount)'],
    'Not_NGShare': ['Percent', 'Number of Not NG per Lisa', 'Not_NGCount / (NGCount + Not_NGCount + PGCount)'],
    'ExpectedCompletion': ['Date', 'Expected completion predction by regression based on the data of last month', ''],
}

In [4]:
kpi_df = pd.DataFrame(
    [
        {"Name": key, "Unit": value[0], "Description": value[1], "Formula": value[2], "LastUpdated": datetime.now()}
        for key, value in output_dict.items()
    ]
)

KPI_Definition.update_table(arguments={'DataFrame': kpi_df, 'db_path': DB_PATH, 'PrimaryKey': 'Name'})

In [5]:
KPI_Definition.query_table(arguments={'db_path': DB_PATH})

,Name,Unit,Formula,Description,LastUpdated
0,ReportAssetLengthKm,Km,,Total length of all the assets in the specifie...,2026-06-12 12:13:41.745221
1,AssetCoveredLengthKm,Km,,Total length of all the assets covered in the ...,2026-06-12 12:13:41.745229
2,DistributionPipeKm,Km,,Total length of all the mains in the report,2026-06-12 12:13:41.745230
3,DistributionPipeCoveredKm,Km,,Total length of all the mains covered in the r...,2026-06-12 12:13:41.745231
4,ServicePipeKm,Km,,Total length of all the services in the report,2026-06-12 12:13:41.745233
5,ServicePipeCoveredKm,Km,,Total length of all the services covered in th...,2026-06-12 12:13:41.745234
6,ReportCount,Report,,Total number of reports in the specified week,2026-06-12 12:13:41.745235
7,DaysCount,Day,,Total number of days worked in the specified week,2026-06-12 12:13:41.745237
8,FOVMain,Percent,DistributionPipeCoveredKm / DistributionPipeKm,Km of mains covered over the total Km of mains,2026-06-12 12:13:41.745238
9,SurveyDurationHours,Hours,,Total hours driven by all cars in the week,2026-06-12 12:13:41.745239
